# ZipEats delivery spread: interval from a summary

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Confidence Interval Estimation for the Variance** · a worked example from the StatNex teaching guide

**The story.** ZipEats is a budget food-delivery app. Customers hate unpredictable deliveries more than slow ones, so ZipEats wants to know how much its orders vary. Only a summary is available: the sample size and standard deviation of delivery time, distance, order total and tip.

**The question.** Can we estimate the spread of delivery times from a two-line summary?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `02_one_sample/one_sample_variance_ci__summary.csv` (2 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Variable,Delivery time (min),Distance (km),Order total ($),Tip ($)
Sample size,50.0,50.0,50.0,50.0
Standard deviation,6.4908,0.6478,3.7695,0.8845
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.6g}".format)

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Delivery time (min)", "Distance (km)", "Order total ($)", "Tip ($)"]
confidence = 95 / 100

## Confidence interval for the variance, one column at a time

In [ ]:
# (n - 1) s² / σ² follows a chi-square distribution with n - 1 degrees of freedom, so
# the interval is built for the variance and printed on the standard-deviation scale
# by taking square roots. It is not symmetric: it reaches further above s than below.
# The interval is exact only for normal data, and unlike an interval for a mean it does
# not become safe as n grows: a long tail or one extreme value makes it too narrow.
# Plot the data first; if the tails are heavy, a robust interval (Bonett's) or a
# bootstrap is the safer report.
results = {}
for name in variables:
    # Summary data: each column holds the sample size and the standard deviation,
    # top to bottom.
    size, sd = pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)[:2]
    n = int(round(size))
    df = n - 1
    variance = sd**2
    # The upper chi-square quantile gives the lower limit, and the lower one the upper.
    lower = df * variance / stats.chi2.ppf(0.5 + confidence / 2, df)
    upper = df * variance / stats.chi2.ppf(0.5 - confidence / 2, df)
    results[name] = {"n": n, "variance": variance, "lower (SD)": np.sqrt(lower),
                     "SD": sd, "upper (SD)": np.sqrt(upper)}

table = pd.DataFrame(results)
print(table)
print(f"\nConfidence level: {confidence:.1%}")

## What the result says

This run uses only two numbers for each measure: the sample size (50 orders) and the standard deviation, the typical distance of one order from the average. For delivery time the standard deviation is 6.49 minutes.

That is enough for a 95% confidence interval, a range of values the data agree with for the true standard deviation of all orders. Built from the chi-square distribution, it runs from 5.42 to 8.09 minutes. The raw orders give the same interval.

So the spread can be estimated from a two-line summary. But two numbers cannot show the shape of the data or an extreme order, so they cannot show whether the interval deserves trust.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.